##Phase 1A: the abstract-section classifier

1. Problem and goal: Predict whether an abstract sentence is background, objective, methods, results, or conclusions.
2. Dataset: PubMed 20k RCT, with numbers preserved.
Inspect the raw files: Understand how abstract IDs, sentences, labels, and abstract boundaries are represented.
3. Parse the data: Preserve abstract IDs and sentence order.
4. Check data quality: Missing values, unexpected labels, and overlap between splits.
5. Explore the training data: Label counts, sentence lengths, and abstract lengths.
Observations and next steps.

Download PubMed 20k RCT.
Parse it into a dataframe.
Explore labels, sentence lengths, class balance, and abstract structure.
Create the PyTorch Dataset and DataLoader.
Build the first section-classification baseline.
Expand it into Biomedical Information Extraction.

In [3]:
!git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git

fatal: destination path 'pubmed-rct' already exists and is not an empty directory.


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path
import pandas as pd


def load_abstract_sentences(file_path):
    rows = []
    abstract_id = None
    sentence_number = 0

    with Path(file_path).open(encoding="utf-8") as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.rstrip("\r\n")

            # A blank line marks the end of an abstract.
            if not line.strip():
                abstract_id = None
                continue

            # An ID line marks the beginning of a new abstract.
            if line.startswith("###"):
                abstract_id = line[3:].strip()
                sentence_number = 0

                if not abstract_id:
                    raise ValueError(
                        f"Missing abstract ID at line {line_number}"
                    )
                continue

            # Every sentence must belong to an abstract.
            if abstract_id is None:
                raise ValueError(
                    f"Sentence without an abstract ID at line {line_number}"
                )

            if "\t" not in line:
                raise ValueError(
                    f"Missing label/text separator at line {line_number}"
                )

            label, sentence = line.split("\t", maxsplit=1)

            rows.append({
                "abstract_id": abstract_id,
                "sentence": sentence,
                "label": label,
                "sentence_number": sentence_number,
            })

            sentence_number += 1

    return pd.DataFrame(
        rows,
        columns=["abstract_id", "sentence", "label", "sentence_number"],
    )

In [6]:
from pathlib import Path

data_dir = Path("pubmed-rct/PubMed_20k_RCT")
train_path = data_dir / "train.txt"
train_df = load_abstract_sentences(data_dir / "train.txt")
validation_df = load_abstract_sentences(data_dir / "dev.txt")
test_df = load_abstract_sentences(data_dir / "test.txt")

train_df.head()

,abstract_id,sentence,label,sentence_number
0,24293578,To investigate the efficacy of 6 weeks of dail...,OBJECTIVE,0
1,24293578,A total of 125 patients with primary knee OA w...,METHODS,1
2,24293578,Outcome measures included pain reduction and i...,METHODS,2
3,24293578,Pain was assessed using the visual analog pain...,METHODS,3
4,24293578,Secondary outcome measures included the Wester...,METHODS,4


In [7]:
print(train_path.resolve())
print("File exists:", train_path.exists())


/content/pubmed-rct/PubMed_20k_RCT/train.txt
File exists: True


In [8]:


splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for name, df in splits.items():
    print(name, '.....')
    print(df.isna().sum())


text_columns = ["abstract_id", "sentence", "label"]

for name, df in splits.items():
    print(name)
    for column in text_columns:
        empty_count = df[column].str.strip().eq("").sum()
        print(column, empty_count)


#number of labels
expected_labels = {
    "BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"
}

for name, df in splits.items():
    observed_labels = set(df["label"].dropna())
    print(name)
    print("Unexpected labels:", observed_labels - expected_labels)
    print("Absent labels:", expected_labels - observed_labels)

#Duplicate records
for name, df in splits.items():
    duplicate_count = df.duplicated(
        subset=["abstract_id", "sentence_number"]
    ).sum()
    print(name, duplicate_count)

train_ids = set(train_df["abstract_id"])
validation_ids = set(validation_df["abstract_id"])
test_ids = set(test_df["abstract_id"])

print("Train / validation overlap:", len(train_ids & validation_ids))
print("Train / test overlap:", len(train_ids & test_ids))
print("Validation / test overlap:", len(validation_ids & test_ids))

#Sentence Order
# Check sentence numbering within each training abstract.
invalid_abstract_ids = []

for abstract_id, group_df in train_df.groupby("abstract_id"):
    expected_order = list(range(len(group_df)))
    actual_order = group_df["sentence_number"].tolist()

    if actual_order != expected_order:
        invalid_abstract_ids.append(abstract_id)

print("Abstracts with invalid numbering:", len(invalid_abstract_ids))
print("First few affected IDs:", invalid_abstract_ids[:5])






train .....
abstract_id        0
sentence           0
label              0
sentence_number    0
dtype: int64
validation .....
abstract_id        0
sentence           0
label              0
sentence_number    0
dtype: int64
test .....
abstract_id        0
sentence           0
label              0
sentence_number    0
dtype: int64
train
abstract_id 0
sentence 0
label 0
validation
abstract_id 0
sentence 0
label 0
test
abstract_id 0
sentence 0
label 0
train
Unexpected labels: set()
Absent labels: set()
validation
Unexpected labels: set()
Absent labels: set()
test
Unexpected labels: set()
Absent labels: set()
train 0
validation 0
test 0
Train / validation overlap: 0
Train / test overlap: 0
Validation / test overlap: 0
Abstracts with invalid numbering: 0
First few affected IDs: []


In [9]:
label_summary = pd.DataFrame({
    "count": train_df["label"].value_counts(),
    "percentage": train_df["label"].value_counts(normalize=True) * 100,
})

print(label_summary)

train_df["word_count"] = train_df["sentence"].str.split().str.len()

print(train_df[["sentence", "word_count"]].head())
print(train_df["word_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))



             count  percentage
label                         
METHODS      59353   32.966563
RESULTS      57953   32.188958
CONCLUSIONS  27168   15.089980
BACKGROUND   21727   12.067874
OBJECTIVE    13839    7.686625
                                            sentence  word_count
0  To investigate the efficacy of 6 weeks of dail...          49
1  A total of 125 patients with primary knee OA w...          27
2  Outcome measures included pain reduction and i...          15
3  Pain was assessed using the visual analog pain...          14
4  Secondary outcome measures included the Wester...          35
count    180040.000000
mean         26.338436
std          15.386976
min           1.000000
50%          23.000000
90%          45.000000
95%          55.000000
99%          80.000000
max         296.000000
Name: word_count, dtype: float64


In [10]:
train_df.nlargest(5, "word_count")[["sentence", "word_count"]]


,sentence,word_count
122524,Patients in the antegrade group achieved bette...,296
71510,There were significant differences in body tem...,275
129612,Significant associations emerged between negle...,246
164330,Eyes undergoing both treatments had improved b...,243
106909,"The scores on sexual self-confidence , sexual ...",242


In [11]:

train_df.nsmallest(55, "word_count")[["sentence", "label", "word_count"]]

# 4921, 6305, 14803, 16686

,sentence,label,word_count
4921,I.,METHODS,1
6305,I.,METHODS,1
14803,I.,METHODS,1
16557,I.,METHODS,1
16686,I.,METHODS,1
34852,I.,METHODS,1
35924,I.,METHODS,1
36807,I.,METHODS,1
49339,I.,METHODS,1
51981,I.,METHODS,1


In [12]:
with train_path.open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if "24306696" in line:
            print(line_number, repr(line))


suspect_id = train_df.loc[62134, "abstract_id"]

print(
    train_df.loc[
        train_df["abstract_id"] == suspect_id,
        ["sentence_number", "label", "sentence"]
    ].to_string(index=False)
)

72524 'CONCLUSIONS\t24306696\n'
 sentence_number       label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              sentence
               0  BACKGROUND                                                                                                                                                                                                                                                                                                    Recombinant human bone morphogenetic protein-2 ( rhBMP-2 ) applied on an absorbable collagen sponge improves open tibial fracture-hea

In [13]:
suspect_id = train_df.loc[4921, "abstract_id"]

print(
    train_df.loc[
        train_df["abstract_id"] == suspect_id,
        ["sentence_number", "label", "sentence"]
    ].to_string(index=False)
)

suspect_id = train_df.loc[6305, "abstract_id"]

print(
    train_df.loc[
        train_df["abstract_id"] == suspect_id,
        ["sentence_number", "label", "sentence"]
    ].to_string(index=False)
)

suspect_id = train_df.loc[16686, "abstract_id"]

print(
    train_df.loc[
        train_df["abstract_id"] == suspect_id,
        ["sentence_number", "label", "sentence"]
    ].to_string(index=False)
)



 sentence_number       label                                                                                                                                                                                                                                                                                                             sentence
               0   OBJECTIVE                                                                                  This study aims to determine the outcome of double-bundle anterior cruciate ligament ( ACL ) reconstruction using an allograft in comparison with ACL reconstruction using a double-bundle autograft or a single-bundle allograft .
               1     METHODS A total of 424 patients who accepted primary ACL reconstructions were divided randomly into three groups : double-bundle technique with autograft ( DB-AU group , n = 154 ) , double-bundle technique with allograft ( DB-AL group , n = 128 ) , and single-bundle technique with allograft ( SB group 

In [14]:
longest_sentence = train_df.loc[
    train_df["word_count"].idxmax(), "sentence"
]

with train_path.open(encoding="utf-8") as file:
    matches = [
        line.rstrip("\r\n")
        for line in file
        if "\t" in line
        and line.rstrip("\r\n").split("\t", 1)[1] == longest_sentence
    ]

print("Exact matching sentence lines:", len(matches))

Exact matching sentence lines: 1


In [15]:
abstract_lengths = train_df.groupby("abstract_id").size()

print(abstract_lengths.describe())

count    15000.000000
mean        12.002667
std          3.072516
min          4.000000
25%         10.000000
50%         12.000000
75%         14.000000
max         31.000000
dtype: float64


In [16]:
majority_label = train_df["label"].value_counts().idxmax()
print(majority_label)

validation_predictions = [majority_label] * len(validation_df)



METHODS


In [17]:
validation_df["label"][0]

'BACKGROUND'

In [18]:
print(validation_df["label"])
err_count = 0
methods_count = 0
for i in range (0, len(validation_df)):
  if (validation_predictions[i]!=validation_df["label"].iloc[i]):
    err_count = err_count + 1
  if(validation_df["label"].iloc[i]=='METHODS'):
    methods_count = methods_count+1
print(err_count, methods_count)
print(err_count/len(validation_df)*100)
accuracy = (len(validation_df) - err_count) / len(validation_df)
precision = methods_count/len(validation_df)
print(f"Validation accuracy: {accuracy:.2%}")
print(f"Precision: {precision:.2%}")


recall = 1.0
methods_f1 = 2 * precision * recall / (precision + recall)

print(f"METHODS F1: {methods_f1:.2%}")

macro_f1 = (methods_f1 + 0 + 0 + 0 + 0) / 5

print(f"Macro-F1: {macro_f1:.2%}")

0         BACKGROUND
1         BACKGROUND
2          OBJECTIVE
3            METHODS
4            METHODS
            ...     
30207        RESULTS
30208        RESULTS
30209        RESULTS
30210    CONCLUSIONS
30211    CONCLUSIONS
Name: label, Length: 30212, dtype: object
20248 9964
67.01972726069113
Validation accuracy: 32.98%
Precision: 32.98%
METHODS F1: 49.60%
Macro-F1: 9.92%


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

# Learn vocabulary and IDF statistics from training sentences,
# then convert those sentences into feature vectors.
X_train = vectorizer.fit_transform(train_df["sentence"])

# Apply the same representation to validation sentences.
X_validation = vectorizer.transform(validation_df["sentence"])

y_train = train_df["label"]
y_validation = validation_df["label"]

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)

Training matrix: (180040, 57996)
Validation matrix: (30212, 57996)


In [20]:
feature_names = vectorizer.get_feature_names_out()

print("Vocabulary size:", len(feature_names))
print("First 20 terms:", feature_names[:20])


first_row = X_train.getrow(0)

print("Sentence:", train_df["sentence"].iloc[0])
print("Label:", train_df["label"].iloc[0])

for column_index, score in zip(first_row.indices, first_row.data):
    print(f"{feature_names[column_index]}: {score:.3f}")

Vocabulary size: 57996
First 20 terms: ['00' '000' '0000' '000000072' '000000752' '000001' '00000154108'
 '00000244' '000002832' '000003103' '000003177' '000003262' '000004511'
 '000005' '00000500' '000005983' '000007972' '00001' '000012012'
 '000012287']
Sentence: To investigate the efficacy of 6 weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at 12 weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .
Label: OBJECTIVE
to: 0.118
investigate: 0.166
the: 0.134
efficacy: 0.128
of: 0.097
weeks: 0.254
daily: 0.145
low: 0.278
dose: 0.134
oral: 0.154
prednisolone: 0.234
in: 0.156
improving: 0.181
pain: 0.127
mobility: 0.216
and: 0.093
systemic: 0.187
grade: 0.176
inflammation: 0.185
short: 0.167
term: 0.152
whether: 0.148
effect: 0.127
would: 0.179
be: 0.113
sustained: 0.191
at: 0.089
12: 0.119
older: 0.169
adults: 0.159
with: 0.065
moderate: 0.161
sev

In [21]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000)

classifier.fit(X_train, y_train)

validation_predictions_lr = classifier.predict(X_validation)

In [22]:
accuracy_lr = (validation_predictions_lr == y_validation).mean()

print(f"Validation accuracy: {accuracy_lr:.2%}")

Validation accuracy: 80.92%


In [23]:
from sklearn.metrics import classification_report

label_order = [
    "BACKGROUND",
    "OBJECTIVE",
    "METHODS",
    "RESULTS",
    "CONCLUSIONS",
]

print(classification_report(
    y_validation,
    validation_predictions_lr,
    labels=label_order,
    digits=3,
    zero_division=0,
))

              precision    recall  f1-score   support

  BACKGROUND      0.626     0.642     0.634      3449
   OBJECTIVE      0.689     0.519     0.592      2376
     METHODS      0.857     0.907     0.881      9964
     RESULTS      0.875     0.877     0.876      9841
 CONCLUSIONS      0.744     0.727     0.736      4582

    accuracy                          0.809     30212
   macro avg      0.758     0.734     0.744     30212
weighted avg      0.806     0.809     0.807     30212



In [24]:
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(
    y_validation,
    validation_predictions_lr,
    labels=label_order,
)

confusion_df = pd.DataFrame(
    matrix,
    index=label_order,
    columns=label_order,
)

confusion_df.index.name = "Actual"
confusion_df.columns.name = "Predicted"

display(confusion_df)

Predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
Actual,,,,,
BACKGROUND,2215,405,317,42,470
OBJECTIVE,707,1233,210,25,201
METHODS,149,75,9035,611,94
RESULTS,45,7,776,8633,380
CONCLUSIONS,420,69,206,555,3332


OBJECTIVE has the lowest recall and F1.

In [25]:
# Keep predictions alongside the corresponding validation sentences.
validation_results = validation_df.copy()
validation_results["predicted_label"] = validation_predictions_lr

# Select actual OBJECTIVE sentences predicted as BACKGROUND.
objective_as_background = validation_results.loc[
    (validation_results["label"] == "OBJECTIVE")
    & (validation_results["predicted_label"] == "BACKGROUND")
]

print("Number of matching errors:", len(objective_as_background))

# Print five reproducibly sampled examples without truncating the text.
examples = objective_as_background.sample(
    n=min(5, len(objective_as_background)),
    random_state=42,
)

for _, row in examples.iterrows():
    print(f"\nAbstract ID: {row['abstract_id']}")
    print(f"Sentence number: {row['sentence_number']}")
    print(f"Actual: {row['label']} | Predicted: {row['predicted_label']}")
    print(row["sentence"])

Number of matching errors: 707

Abstract ID: 24963091
Sentence number: 2
Actual: OBJECTIVE | Predicted: BACKGROUND
Here , we addressed the question of whether NMES can modify upper esophageal sphincter ( UES ) dynamics .

Abstract ID: 24565089
Sentence number: 1
Actual: OBJECTIVE | Predicted: BACKGROUND
Several analgesic techniques have been advocated .

Abstract ID: 25416724
Sentence number: 2
Actual: OBJECTIVE | Predicted: BACKGROUND
However , MetS has also been associated with higher estradiol ( E2 ) levels , which is bone protective .

Abstract ID: 24799054
Sentence number: 1
Actual: OBJECTIVE | Predicted: BACKGROUND
We report the activity and safety results of biweekly combination of trastuzumab , docetaxel and gemcitabine as first-line therapy in HER2-positive advanced breast cancer .

Abstract ID: 24941010
Sentence number: 1
Actual: OBJECTIVE | Predicted: BACKGROUND
We hypothesize that strict maintenance of normoglycemia during cardiac surgery preserves postoperative cognitive f

In [26]:
def tokenize(sentence):
    return sentence.lower().split()

example_sentence = train_df["sentence"].iloc[0]
example_tokens = tokenize(example_sentence)

print(example_sentence)
print(example_tokens)
print("Number of tokens:", len(example_tokens))

To investigate the efficacy of 6 weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at 12 weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .
['to', 'investigate', 'the', 'efficacy', 'of', '6', 'weeks', 'of', 'daily', 'low-dose', 'oral', 'prednisolone', 'in', 'improving', 'pain', ',', 'mobility', ',', 'and', 'systemic', 'low-grade', 'inflammation', 'in', 'the', 'short', 'term', 'and', 'whether', 'the', 'effect', 'would', 'be', 'sustained', 'at', '12', 'weeks', 'in', 'older', 'adults', 'with', 'moderate', 'to', 'severe', 'knee', 'osteoarthritis', '(', 'oa', ')', '.']
Number of tokens: 49


In [27]:
vocab = {"<PAD>": 0, "<UNK>": 1}

for sentence in train_df["sentence"]:
    for token in tokenize(sentence):
        if token not in vocab:
            vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:15])

Vocabulary size: 102910
[('<PAD>', 0), ('<UNK>', 1), ('to', 2), ('investigate', 3), ('the', 4), ('efficacy', 5), ('of', 6), ('6', 7), ('weeks', 8), ('daily', 9), ('low-dose', 10), ('oral', 11), ('prednisolone', 12), ('in', 13), ('improving', 14)]


In [28]:
example_ids = [
    vocab.get(token, vocab["<UNK>"])
    for token in example_tokens
]

print(example_ids)
print("Number of IDs:", len(example_ids))

[2, 3, 4, 5, 6, 7, 8, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 16, 18, 19, 20, 21, 13, 4, 22, 23, 18, 24, 4, 25, 26, 27, 28, 29, 30, 8, 13, 31, 32, 33, 34, 2, 35, 36, 37, 38, 39, 40, 41]
Number of IDs: 49


In [29]:
label_to_id = {
    "BACKGROUND": 0,
    "OBJECTIVE": 1,
    "METHODS": 2,
    "RESULTS": 3,
    "CONCLUSIONS": 4,
}

id_to_label = {
    label_id: label
    for label, label_id in label_to_id.items()
}

print(id_to_label)

{0: 'BACKGROUND', 1: 'OBJECTIVE', 2: 'METHODS', 3: 'RESULTS', 4: 'CONCLUSIONS'}


In [30]:
sentence = train_df["sentence"].iloc[0]
label = train_df["label"].iloc[0]

token_ids = [
    vocab.get(token, vocab["<UNK>"])
    for token in tokenize(sentence)
]
label_id = label_to_id[label]

print("Token IDs:", token_ids)
print("Label:", label)
print("Label ID:", label_id)

Token IDs: [2, 3, 4, 5, 6, 7, 8, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 16, 18, 19, 20, 21, 13, 4, 22, 23, 18, 24, 4, 25, 26, 27, 28, 29, 30, 8, 13, 31, 32, 33, 34, 2, 35, 36, 37, 38, 39, 40, 41]
Label: OBJECTIVE
Label ID: 1


In [31]:
import torch
from torch.utils.data import Dataset


class AbstractSentenceDataset(Dataset):
    def __init__(self, dataframe, vocab, label_to_id):
        self.sentences = dataframe["sentence"].tolist()
        self.labels = dataframe["label"].tolist()
        self.vocab = vocab
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):
        sentence = self.sentences[index]
        label = self.labels[index]

        token_ids = [
            self.vocab.get(token, self.vocab["<UNK>"])
            for token in tokenize(sentence)
        ]
        label_id = self.label_to_id[label]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(label_id, dtype=torch.long),
        )

In [32]:
train_dataset = AbstractSentenceDataset(
    train_df, vocab, label_to_id
)

validation_dataset = AbstractSentenceDataset(
    validation_df, vocab, label_to_id
)

In [33]:
token_ids, label_id = train_dataset[0]

print("Number of examples:", len(train_dataset))
print("Token IDs:", token_ids)
print("Token tensor shape:", token_ids.shape)
print("Label ID:", label_id.item())
print("Label:", id_to_label[label_id.item()])

Number of examples: 180040
Token IDs: tensor([ 2,  3,  4,  5,  6,  7,  8,  6,  9, 10, 11, 12, 13, 14, 15, 16, 17, 16,
        18, 19, 20, 21, 13,  4, 22, 23, 18, 24,  4, 25, 26, 27, 28, 29, 30,  8,
        13, 31, 32, 33, 34,  2, 35, 36, 37, 38, 39, 40, 41])
Token tensor shape: torch.Size([49])
Label ID: 1
Label: OBJECTIVE


In [34]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader


def collate_batch(batch):
    # Each example contains (token_ids, label_id).
    token_sequences, labels = zip(*batch)

    padded_tokens = pad_sequence(
        token_sequences,
        batch_first=True,
        padding_value=vocab["<PAD>"],
    )

    labels = torch.stack(labels)

    return padded_tokens, labels

In [35]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_batch,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_batch,
)

In [36]:
batch_tokens, batch_labels = next(iter(train_loader))

print("Token shape:", batch_tokens.shape)
print("Label shape:", batch_labels.shape)

print("First sentence IDs:", batch_tokens[0])
print("Its label ID:", batch_labels[0].item())
print("Its actual length:", (batch_tokens[0] != vocab["<PAD>"]).sum().item())

Token shape: torch.Size([64, 110])
Label shape: torch.Size([64])
First sentence IDs: tensor([  266,   196,   147,   145, 25548,   165,   200,     2,    27,   205,
         1200,   108,    57, 13226,  1877,     4,  4008,   510,  4502,     2,
         1780,  3637,    41,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
Its label ID: 4
Its actual length: 23


In [37]:
from torch import nn


class AbstractClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, pad_id):
        super().__init__()

        self.pad_id = pad_id

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)

        # True for real tokens; False for padding.
        mask = (token_ids != self.pad_id).unsqueeze(-1)

        # Sum real-token vectors, then divide by real-token count.
        summed = (embeddings * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        sentence_vectors = summed / lengths

        return self.classifier(sentence_vectors)

In [38]:
model = AbstractClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    num_classes=len(label_to_id),
    pad_id=vocab["<PAD>"],
)

with torch.no_grad():
    scores = model(batch_tokens)

print("Output shape:", scores.shape)
print("First sentence's scores:", scores[0])

batch_tokens, batch_labels = next(iter(train_loader))

with torch.no_grad():
    scores = model(batch_tokens)

print("Input shape:", batch_tokens.shape)
print("Label shape:", batch_labels.shape)
print("Output shape:", scores.shape)

Output shape: torch.Size([64, 5])
First sentence's scores: tensor([-0.0709,  0.0127,  0.0076, -0.0820,  0.1739])
Input shape: torch.Size([64, 53])
Label shape: torch.Size([64])
Output shape: torch.Size([64, 5])


In [39]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [40]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for tokens, labels in loader:
        optimizer.zero_grad()

        scores = model(tokens)
        loss = loss_fn(scores, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (scores.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

In [41]:
def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for tokens, labels in loader:
            scores = model(tokens)
            loss = loss_fn(scores, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (scores.argmax(dim=1) == labels).sum().item()
            total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

In [42]:
# history = []

# for epoch in range(5, 6):
#     train_loss, train_accuracy = train_one_epoch(
#         model, train_loader, loss_fn, optimizer
#     )

#     val_loss, val_accuracy = evaluate(
#         model, validation_loader, loss_fn
#     )

#     history.append({
#         "epoch": epoch,
#         "train_loss": train_loss,
#         "train_accuracy": train_accuracy,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#     })

#     print(
#         f"Epoch {epoch} | "
#         f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.2%} | "
#         f"Validation loss: {val_loss:.4f}, accuracy: {val_accuracy:.2%}"
#     )

In [43]:
# checkpoint = {
#     "model_state_dict": model.state_dict(),
#     "optimizer_state_dict": optimizer.state_dict(),
#     "vocab": vocab,
#     "label_to_id": label_to_id,
#     "embedding_dim": model.embedding.embedding_dim,
#     "pad_id": model.pad_id,
#     "epochs_completed": 4,
#     "validation_accuracy": val_accuracy,
#     "validation_loss": val_loss,
#     "tokenizer": "lowercase_then_whitespace_split",
# }

# torch.save(checkpoint, "abstract_classifier_epoch4.pt")

In [44]:
# from google.colab import files

# files.download("abstract_classifier_epoch4.pt")

In [45]:
checkpoint_dir = Path(
    "/content/drive/MyDrive/Portfolio_Project/Abstract_Classifier/checkpoints"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

restored_path = checkpoint_dir / "abstract_classifier_epoch4.pt"

checkpoint = torch.load(
    restored_path,
    map_location="cpu",
    weights_only=True,
)

restored_model = AbstractClassifier(
    vocab_size=len(checkpoint["vocab"]),
    embedding_dim=checkpoint["embedding_dim"],
    num_classes=len(checkpoint["label_to_id"]),
    pad_id=checkpoint["pad_id"],
)

restored_model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [46]:
restored_loss, restored_accuracy = evaluate(
    restored_model, validation_loader, loss_fn
)

print(f"Restored validation loss: {restored_loss:.4f}")
print(f"Restored validation accuracy: {restored_accuracy:.2%}")

Restored validation loss: 0.5384
Restored validation accuracy: 80.71%


In [47]:
saved_vocab = checkpoint["vocab"]
saved_id_to_label = {
    label_id: label
    for label, label_id in checkpoint["label_to_id"].items()
}


def predict_section(sentence):
    tokens = tokenize(sentence)

    if not tokens:
        raise ValueError("Please provide a non-empty sentence.")

    token_ids = [
        saved_vocab.get(token, saved_vocab["<UNK>"])
        for token in tokens
    ]

    # Shape: [1, number_of_tokens] — a batch of one sentence.
    inputs = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)

    restored_model.eval()

    with torch.no_grad():
        scores = restored_model(inputs)
        predicted_id = scores.argmax(dim=1).item()

    return saved_id_to_label[predicted_id]

In [48]:
sentence = "We randomly assigned 350 patients to treatment or placebo."
print(predict_section(sentence))

METHODS


In [49]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for batch_tokens, batch_labels in validation_loader:
        scores = restored_model(batch_tokens)
        predictions = scores.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

In [50]:
from sklearn.metrics import classification_report

class_ids = list(range(len(label_to_id)))
class_names = [id_to_label[i] for i in class_ids]

print(classification_report(
    all_labels,
    all_predictions,
    labels=class_ids,
    target_names=class_names,
    digits=3,
    zero_division=0,
))

              precision    recall  f1-score   support

  BACKGROUND      0.645     0.586     0.614      3449
   OBJECTIVE      0.633     0.574     0.602      2376
     METHODS      0.880     0.882     0.881      9964
     RESULTS      0.849     0.894     0.871      9841
 CONCLUSIONS      0.747     0.743     0.745      4582

    accuracy                          0.807     30212
   macro avg      0.751     0.736     0.743     30212
weighted avg      0.803     0.807     0.805     30212



In [51]:
# from sklearn.metrics import f1_score


# tfidf_macro_f1 = f1_score(
#     y_validation,
#     validation_predictions_lr,
#     average="macro",
# )

# print(f"TF-IDF macro-F1: {tfidf_macro_f1:.3f}")

In [52]:
from torch import nn


class CNNAbstractClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_filters,
        num_classes,
        pad_id,
    ):
        super().__init__()

        self.pad_id = pad_id

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id,
        )

        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=num_filters,
            kernel_size=3,
            padding=1,
        )

        self.activation = nn.ReLU()
        self.classifier = nn.Linear(num_filters, num_classes)

    def forward(self, token_ids):
        # [B, L, D]
        embeddings = self.embedding(token_ids)

        # Conv1d expects [B, D, L].
        embeddings = embeddings.transpose(1, 2)

        # Each filter produces one response per token position.
        # Shape: [B, num_filters, L]
        features = self.activation(self.conv(embeddings))

        # Exclude positions centred on padding from max pooling.
        # Shape: [B, 1, L]; broadcasts across all filters.
        mask = (token_ids != self.pad_id).unsqueeze(1)
        features = features.masked_fill(~mask, float("-inf"))

        # Maximum across positions, separately for each filter.
        # Shape: [B, num_filters]
        sentence_features = features.max(dim=2).values

        # Shape: [B, num_classes]
        return self.classifier(sentence_features)

In [53]:
cnn_model = CNNAbstractClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    num_filters=32,
    num_classes=len(label_to_id),
    pad_id=vocab["<PAD>"],
)

batch_tokens, batch_labels = next(iter(train_loader))

with torch.no_grad():
    scores = cnn_model(batch_tokens)

print("Input shape:", batch_tokens.shape)
print("Output shape:", scores.shape)

Input shape: torch.Size([64, 71])
Output shape: torch.Size([64, 5])


In [54]:
from pathlib import Path

checkpoint_dir = Path(
    "/content/drive/MyDrive/Portfolio_Project/Abstract_Classifier/checkpoints"
)

In [55]:
from copy import deepcopy

#from zero

cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

cnn_history = []
best_cnn_loss = float("inf")
best_cnn_checkpoint = None

In [56]:
from copy import deepcopy

#load prev model

checkpoint = torch.load(
    checkpoint_dir / "cnn_latest.pt",
    map_location="cpu",
    weights_only=True,
)

# Restore the exact mappings used during training.
vocab = checkpoint["vocab"]
label_to_id = checkpoint["label_to_id"]

cnn_model = CNNAbstractClassifier(
    vocab_size=len(vocab),
    embedding_dim=checkpoint["embedding_dim"],
    num_filters=checkpoint["num_filters"],
    num_classes=len(label_to_id),
    pad_id=checkpoint["pad_id"],
)
cnn_model.load_state_dict(checkpoint["model_state_dict"])

cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
cnn_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

loss_fn = nn.CrossEntropyLoss()

cnn_history = checkpoint["history"]
best_cnn_loss = checkpoint["best_validation_loss"]

best_cnn_checkpoint = torch.load(
    checkpoint_dir / "cnn_best.pt",
    map_location="cpu",
    weights_only=True,
)

start_epoch = checkpoint["epochs_completed"] + 1

In [57]:
# #additional_epochs = 2

# #for epoch in range(start_epoch, start_epoch + additional_epochs):
#     # Same training, validation, and checkpoint-saving body.
# for epoch in range(1, 6):
#     train_loss, train_accuracy = train_one_epoch(
#         cnn_model, train_loader, loss_fn, cnn_optimizer
#     )

#     val_loss, val_accuracy = evaluate(
#         cnn_model, validation_loader, loss_fn
#     )

#     cnn_history.append({
#         "epoch": epoch,
#         "train_loss": train_loss,
#         "train_accuracy": train_accuracy,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#     })

#     improved = val_loss < best_cnn_loss

#     if improved:
#         best_cnn_loss = val_loss

#         best_cnn_checkpoint = {
#             "model_state_dict": deepcopy(cnn_model.state_dict()),
#             "optimizer_state_dict": deepcopy(cnn_optimizer.state_dict()),
#             "vocab": vocab.copy(),
#             "label_to_id": label_to_id.copy(),
#             "embedding_dim": cnn_model.embedding.embedding_dim,
#             "num_filters": cnn_model.conv.out_channels,
#             "kernel_size": cnn_model.conv.kernel_size[0],
#             "padding": cnn_model.conv.padding[0],
#             "pad_id": cnn_model.pad_id,
#             "epochs_completed": epoch,
#             "validation_loss": val_loss,
#             "validation_accuracy": val_accuracy,
#             "tokenizer": "lowercase_then_whitespace_split",
#         }

#         torch.save(best_cnn_checkpoint, "cnn_best.pt")

#     latest_checkpoint = {
#       **best_cnn_checkpoint,  # Reuse vocabulary and architecture metadata.
#       "model_state_dict": cnn_model.state_dict(),
#       "optimizer_state_dict": cnn_optimizer.state_dict(),
#       "epochs_completed": epoch,
#       "validation_loss": val_loss,
#       "validation_accuracy": val_accuracy,
#       "best_validation_loss": best_cnn_loss,
#       "history": cnn_history,
#     }

#     torch.save(latest_checkpoint, "cnn_latest.pt")

#     print(
#         f"Epoch {epoch} | "
#         f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.2%} | "
#         f"Validation loss: {val_loss:.4f}, accuracy: {val_accuracy:.2%}"
#         + (" | Best saved" if improved else ""),
#         flush=True,
#     )

In [58]:
# #temp


# latest_checkpoint = {
#     **best_cnn_checkpoint,
#     "model_state_dict": cnn_model.state_dict(),
#     "optimizer_state_dict": cnn_optimizer.state_dict(),
#     "epochs_completed": epoch,
#     "validation_loss": val_loss,
#     "validation_accuracy": val_accuracy,
#     "best_validation_loss": best_cnn_loss,
#     "history": cnn_history,
# }

# torch.save(latest_checkpoint, "cnn_latest.pt")

In [59]:
# from google.colab import files

# files.download("cnn_best.pt")
# files.download("cnn_latest.pt")

In [60]:
cnn_model.load_state_dict(
    best_cnn_checkpoint["model_state_dict"]
)

print("Selected epoch:", best_cnn_checkpoint["epochs_completed"])

Selected epoch: 3


In [61]:
cnn_predictions = []
cnn_labels = []

cnn_model.eval()

with torch.no_grad():
    for tokens, labels in validation_loader:
        scores = cnn_model(tokens)

        cnn_predictions.extend(scores.argmax(dim=1).cpu().tolist())
        cnn_labels.extend(labels.cpu().tolist())

print(classification_report(
    cnn_labels,
    cnn_predictions,
    labels=class_ids,
    target_names=class_names,
    digits=3,
    zero_division=0,
))

              precision    recall  f1-score   support

  BACKGROUND      0.601     0.714     0.653      3449
   OBJECTIVE      0.761     0.471     0.582      2376
     METHODS      0.876     0.905     0.890      9964
     RESULTS      0.882     0.861     0.872      9841
 CONCLUSIONS      0.724     0.747     0.735      4582

    accuracy                          0.811     30212
   macro avg      0.769     0.740     0.746     30212
weighted avg      0.814     0.811     0.809     30212



In [62]:
cnn_confusion = pd.DataFrame(
    confusion_matrix(
        cnn_labels,
        cnn_predictions,
        labels=class_ids,
    ),
    index=class_names,
    columns=class_names,
)

cnn_confusion.index.name = "Actual"
cnn_confusion.columns.name = "Predicted"

display(cnn_confusion)

Predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
Actual,,,,,
BACKGROUND,2464,283,209,45,448
OBJECTIVE,901,1119,153,22,181
METHODS,207,49,9020,557,131
RESULTS,47,4,767,8478,545
CONCLUSIONS,481,16,149,512,3424


In [63]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from torch import nn

##########CLASS WEIGHTED CROSS ENTROPY##############


import torch
from torch import nn
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")


checkpoint_dir = Path(
    "/content/drive/MyDrive/Portfolio_Project/Abstract_Classifier/checkpoints"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Keep weights aligned with the numerical class IDs.
class_names = [
    label for label, class_id in
    sorted(label_to_id.items(), key=lambda item: item[1])
]

counts = train_df["label"].value_counts().reindex(class_names)

# Smaller classes receive larger weights.
class_weights = torch.tensor(
    (len(train_df) / (len(class_names) * counts)).to_numpy(),
    dtype=torch.float32,
)

print("Class weights:")
for name, weight in zip(class_names, class_weights):
    print(f"{name}: {weight.item():.3f}")

torch.manual_seed(42)

weighted_cnn = CNNAbstractClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    num_filters=32,
    num_classes=len(label_to_id),
    pad_id=vocab["<PAD>"],
)

weighted_optimizer = torch.optim.Adam(
    weighted_cnn.parameters(), lr=0.001
)

# Sum weighted losses so we can calculate the correct weighted mean.
weighted_loss_sum = nn.CrossEntropyLoss(
    weight=class_weights, reduction="sum"
)

# Use ordinary validation loss for comparison with earlier runs.
validation_loss_fn = nn.CrossEntropyLoss()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class weights:
BACKGROUND: 1.657
OBJECTIVE: 2.602
METHODS: 0.607
RESULTS: 0.621
CONCLUSIONS: 1.325


In [64]:
def train_weighted_epoch(model, loader, optimizer):
    model.train()

    total_loss_sum = 0.0
    total_weight = 0.0
    total_correct = 0
    total_examples = 0

    for tokens, labels in loader:
        optimizer.zero_grad()

        scores = model(tokens)

        loss_sum = weighted_loss_sum(scores, labels)
        weight_sum = class_weights[labels].sum()
        loss = loss_sum / weight_sum

        loss.backward()
        optimizer.step()

        total_loss_sum += loss_sum.item()
        total_weight += weight_sum.item()
        total_correct += (scores.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)

    return (
        total_loss_sum / total_weight,
        total_correct / total_examples,
    )

In [65]:
# weighted_history = []
# best_val_loss = float("inf")

# for epoch in range(1, 6):
#     train_loss, train_accuracy = train_weighted_epoch(
#         weighted_cnn, train_loader, weighted_optimizer
#     )

#     val_loss, val_accuracy = evaluate(
#         weighted_cnn, validation_loader, validation_loss_fn
#     )

#     weighted_history.append({
#         "epoch": epoch,
#         "weighted_train_loss": train_loss,
#         "train_accuracy": train_accuracy,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#     })

#     improved = val_loss < best_val_loss
#     if improved:
#         best_val_loss = val_loss

#     checkpoint = {
#         "model_state_dict": weighted_cnn.state_dict(),
#         "optimizer_state_dict": weighted_optimizer.state_dict(),
#         "vocab": vocab,
#         "label_to_id": label_to_id,
#         "embedding_dim": 64,
#         "num_filters": 32,
#         "kernel_size": 3,
#         "padding": 1,
#         "pad_id": vocab["<PAD>"],
#         "class_weights": class_weights,
#         "epochs_completed": epoch,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#         "best_validation_loss": best_val_loss,
#         "history": weighted_history,
#         "tokenizer": "lowercase_then_whitespace_split",
#     }

#     # Written immediately: no deepcopy needed.
#     torch.save(
#         checkpoint, checkpoint_dir / "cnn_weighted_latest.pt"
#     )

#     if improved:
#         torch.save(
#             checkpoint, checkpoint_dir / "cnn_weighted_best.pt"
#         )

#     print(
#         f"Epoch {epoch} | "
#         f"Weighted train loss: {train_loss:.4f}, "
#         f"accuracy: {train_accuracy:.2%} | "
#         f"Validation loss: {val_loss:.4f}, "
#         f"accuracy: {val_accuracy:.2%}"
#         + (" | Best saved" if improved else ""),
#         flush=True,
#     )

In [66]:
from sklearn.metrics import classification_report

best_checkpoint = torch.load(
    checkpoint_dir / "cnn_weighted_best.pt",
    map_location="cpu",
    weights_only=True,
)

weighted_cnn.load_state_dict(best_checkpoint["model_state_dict"])
weighted_cnn.eval()

predictions = []
actual_labels = []

with torch.no_grad():
    for tokens, labels in validation_loader:
        scores = weighted_cnn(tokens)
        predictions.extend(scores.argmax(dim=1).tolist())
        actual_labels.extend(labels.tolist())

print("Selected epoch:", best_checkpoint["epochs_completed"])

print(classification_report(
    actual_labels,
    predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=3,
    zero_division=0,
))

Selected epoch: 3
              precision    recall  f1-score   support

  BACKGROUND      0.599     0.634     0.616      3449
   OBJECTIVE      0.613     0.601     0.607      2376
     METHODS      0.888     0.873     0.880      9964
     RESULTS      0.884     0.827     0.855      9841
 CONCLUSIONS      0.677     0.772     0.721      4582

    accuracy                          0.794     30212
   macro avg      0.732     0.742     0.736     30212
weighted avg      0.800     0.794     0.796     30212



##CNN vs weighted CNN

In this run, inverse-frequency class weighting increased OBJECTIVE recall from 47.1% to 60.1%, but reduced precision and overall macro-F1. The unweighted CNN remained the stronger overall baseline. Different random initializations limit attributing the entire difference to weighting.

In [67]:
# Assumes validation_loader used shuffle=False and validation_df order.
results = validation_df.copy()

id_to_label = {
    class_id: label
    for label, class_id in label_to_id.items()
}

assert len(predictions) == len(results)
assert actual_labels == [
    label_to_id[label] for label in results["label"]
]

results["predicted_label"] = [
    id_to_label[prediction] for prediction in predictions
]

errors = results.loc[
    (results["label"] == "OBJECTIVE")
    & (results["predicted_label"] == "BACKGROUND")
]

print("OBJECTIVE → BACKGROUND errors:", len(errors))

examples = errors.sample(n=min(5, len(errors)), random_state=42)

for _, error in examples.iterrows():
    print(f"\n{'=' * 70}")
    print(f"Abstract ID: {error['abstract_id']}")

    abstract = results.loc[
        results["abstract_id"] == error["abstract_id"]
    ].sort_values("sentence_number")

    for _, row in abstract.iterrows():
        marker = (
            ">>>"
            if row["sentence_number"] == error["sentence_number"]
            else "   "
        )

        print(
            f"\n{marker} Sentence {row['sentence_number']} | "
            f"Actual: {row['label']} | "
            f"Predicted: {row['predicted_label']}"
        )
        print(row["sentence"])

OBJECTIVE → BACKGROUND errors: 681

Abstract ID: 25286782

>>> Sentence 0 | Actual: OBJECTIVE | Predicted: BACKGROUND
Administration of many drugs including magnesium sulfate ( MS ) has considerable influences on pregnancy outcomes .

    Sentence 1 | Actual: OBJECTIVE | Predicted: OBJECTIVE
The present study investigates the effects of MS administration on reaching the active phase of labor in women with premature rupture of membrane ( PROM ) and subsequent fetal complications .

    Sentence 2 | Actual: METHODS | Predicted: METHODS
A double blind , randomized , placebo-controlled trial was performed among primipara women referred to the PROM center in Tehran , Iran between March 2010 and August 2012 .

    Sentence 3 | Actual: METHODS | Predicted: RESULTS
Patients were equally allocated into two groups ; the intervention group who received MS ( n = 46 ) and the control ( placebo ) group ( n = 46 ) .

    Sentence 4 | Actual: METHODS | Predicted: METHODS
Both groups received a cortico

In [68]:
#test

# Two examples: one METHODS, one OBJECTIVE.
example_scores = torch.zeros(2, len(label_to_id))

example_labels = torch.tensor([
    label_to_id["METHODS"],
    label_to_id["OBJECTIVE"],
])

# Calculate a separate loss for each example.
plain_losses = nn.CrossEntropyLoss(reduction="none")(
    example_scores, example_labels
)

weighted_losses = nn.CrossEntropyLoss(
    weight=class_weights,
    reduction="none",
)(example_scores, example_labels)

print("Unweighted losses:", plain_losses)
print("Class weights:", class_weights[example_labels])
print("Weighted losses:", weighted_losses)

Unweighted losses: tensor([1.6094, 1.6094])
Class weights: tensor([0.6067, 2.6019])
Weighted losses: tensor([0.9764, 4.1876])


In [69]:
example = errors.iloc[0]

abstract = results.loc[
    results["abstract_id"] == example["abstract_id"]
].sort_values("sentence_number")

print("Misclassified sentence:")
print(example["sentence"])

print("\nFull abstract:")
for _, row in abstract.iterrows():
    marker = ">>>" if row["sentence_number"] == example["sentence_number"] else ""

    print(
        f"\n{marker} Sentence {row['sentence_number']} "
        f"| Actual: {row['label']} "
        f"| Predicted: {row['predicted_label']}"
    )
    print(row["sentence"])

Misclassified sentence:
To estimate age at attaining Tanner stages in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .

Full abstract:

>>> Sentence 0 | Actual: OBJECTIVE | Predicted: BACKGROUND
To estimate age at attaining Tanner stages in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .

 Sentence 1 | Actual: METHODS | Predicted: METHODS
Observational analysis within a randomized trial .

 Sentence 2 | Actual: METHODS | Predicted: METHODS
Tanner staging was assessed every 24 weeks from 10 years of age , menarche every 12 weeks and height every 4-6 weeks .

 Sentence 3 | Actual: METHODS | Predicted: METHODS
Age at attaining different Tanner stages was estimated using normal interval regression , considering predic

In [70]:
examples = [
    example["sentence"],
    "To estimate the timing of puberty and investigate predictors of delay .",
    "To estimate outcomes and investigate predictors .",
]

weighted_cnn.eval()

with torch.no_grad():
    for sentence in examples:
        ids = [
            vocab.get(token, vocab["<UNK>"])
            for token in tokenize(sentence)
        ]
        inputs = torch.tensor(ids, dtype=torch.long).unsqueeze(0)

        predicted_id = weighted_cnn(inputs).argmax(dim=1).item()

        print(id_to_label[predicted_id], "|", sentence)

BACKGROUND | To estimate age at attaining Tanner stages in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .
OBJECTIVE | To estimate the timing of puberty and investigate predictors of delay .
OBJECTIVE | To estimate outcomes and investigate predictors .


In [71]:
original = example["sentence"]

probes = {
    "Original": original,
    "Replace opening": original.replace(
        "To estimate", "This study aimed to estimate", 1
    ),
    "Simplify one technical phrase": original.replace(
        "Tanner stages", "stages of puberty"
    ),
    "Simplify population description": original.replace(
        "Ugandan/Zimbabwean HIV-infected children",
        "children with HIV"
    ),
}

weighted_cnn.eval()

with torch.no_grad():
    for name, sentence in probes.items():
        ids = [
            vocab.get(token, vocab["<UNK>"])
            for token in tokenize(sentence)
        ]
        inputs = torch.tensor(ids, dtype=torch.long).unsqueeze(0)

        predicted_id = weighted_cnn(inputs).argmax(dim=1).item()

        print(f"{name}: {id_to_label[predicted_id]}")
        print(sentence)
        print()

Original: BACKGROUND
To estimate age at attaining Tanner stages in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .

Replace opening: BACKGROUND
This study aimed to estimate age at attaining Tanner stages in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .

Simplify one technical phrase: BACKGROUND
To estimate age at attaining stages of puberty in Ugandan/Zimbabwean HIV-infected children initiating antiretroviral therapy ( ART ) in older childhood and investigate predictors of delayed puberty , particularly age at ART initiation .

Simplify population description: BACKGROUND
To estimate age at attaining Tanner stages in children with HIV initiating antiretroviral therapy ( ART ) in older childhood and investigate p

In [72]:
example = errors.iloc[30]

abstract = results.loc[
    results["abstract_id"] == example["abstract_id"]
].sort_values("sentence_number")

print("Misclassified sentence:")
print(example["sentence"])

print("\nFull abstract:")
for _, row in abstract.iterrows():
    marker = ">>>" if row["sentence_number"] == example["sentence_number"] else ""

    print(
        f"\n{marker} Sentence {row['sentence_number']} "
        f"| Actual: {row['label']} "
        f"| Predicted: {row['predicted_label']}"
    )
    print(row["sentence"])

Misclassified sentence:
Community-based efforts to prevent adolescent problem behaviors are essential to promote public health and achieve collective impact community wide .

Full abstract:

>>> Sentence 0 | Actual: OBJECTIVE | Predicted: BACKGROUND
Community-based efforts to prevent adolescent problem behaviors are essential to promote public health and achieve collective impact community wide .

 Sentence 1 | Actual: OBJECTIVE | Predicted: OBJECTIVE
OBJECTIVE To test whether the Communities That Care ( CTC ) prevention system reduced levels of risk and adolescent problem behaviors community wide 8 years after implementation of CTC .

 Sentence 2 | Actual: METHODS | Predicted: METHODS
A community-randomized trial was performed in 24 small towns in 7 states , matched within state , assigned randomly to a control or intervention group in 2003 .

 Sentence 3 | Actual: METHODS | Predicted: METHODS
All fifth-grade students attending public schools in study communities in 2003-2004 who rece

In [73]:
example = errors.iloc[33]

abstract = results.loc[
    results["abstract_id"] == example["abstract_id"]
].sort_values("sentence_number")

print("Misclassified sentence:")
print(example["sentence"])

print("\nFull abstract:")
for _, row in abstract.iterrows():
    marker = ">>>" if row["sentence_number"] == example["sentence_number"] else ""

    print(
        f"\n{marker} Sentence {row['sentence_number']} "
        f"| Actual: {row['label']} "
        f"| Predicted: {row['predicted_label']}"
    )
    print(row["sentence"])

Misclassified sentence:
A common challenge for the utilization of fNIRS in these areas is a stable and reliable investigation of the spatio-temporal hemodynamic patterns .

Full abstract:

 Sentence 0 | Actual: OBJECTIVE | Predicted: OBJECTIVE
Functional near-infrared spectroscopy ( fNIRS ) is an emerging technique for the in vivo assessment of functional activity of the cerebral cortex as well as in the field of brain-computer interface ( BCI ) research .

>>> Sentence 1 | Actual: OBJECTIVE | Predicted: BACKGROUND
A common challenge for the utilization of fNIRS in these areas is a stable and reliable investigation of the spatio-temporal hemodynamic patterns .

 Sentence 2 | Actual: OBJECTIVE | Predicted: CONCLUSIONS
However , the recorded patterns may be influenced and superimposed by signals generated from physiological processes , resulting in an inaccurate estimation of the cortical activity .

 Sentence 3 | Actual: OBJECTIVE | Predicted: BACKGROUND
Up to now only a few studies hav

In [74]:
examples = [
    example["sentence"],
    "Community-based efforts to prevent adolescent problem behaviors.",
    "To promote public health and achieve collective impact community wide preventing adolescent problem behaviors are essential .",
    "To promote public health and achieve collective impact community wide, preventing adolescent problem behaviors are essential .",
    "To promote public health and achieve collective impact community wide, preventing adolescent problem behaviors using community-based efforts are essential .",
    "Community-based efforts to prevent adolescent problem behaviors are essential in promoting public health .",
]

weighted_cnn.eval()

with torch.no_grad():
    for sentence in examples:
        ids = [
            vocab.get(token, vocab["<UNK>"])
            for token in tokenize(sentence)
        ]
        inputs = torch.tensor(ids, dtype=torch.long).unsqueeze(0)

        predicted_id = weighted_cnn(inputs).argmax(dim=1).item()

        print(id_to_label[predicted_id], "|", sentence)

BACKGROUND | A common challenge for the utilization of fNIRS in these areas is a stable and reliable investigation of the spatio-temporal hemodynamic patterns .
BACKGROUND | Community-based efforts to prevent adolescent problem behaviors.
OBJECTIVE | To promote public health and achieve collective impact community wide preventing adolescent problem behaviors are essential .
BACKGROUND | To promote public health and achieve collective impact community wide, preventing adolescent problem behaviors are essential .
BACKGROUND | To promote public health and achieve collective impact community wide, preventing adolescent problem behaviors using community-based efforts are essential .
BACKGROUND | Community-based efforts to prevent adolescent problem behaviors are essential in promoting public health .


In [75]:
# Longer headings come first so "BACKGROUND AND OBJECTIVE"
# is matched before "BACKGROUND".
headings = [
    "BACKGROUND AND OBJECTIVES",
    "BACKGROUND AND OBJECTIVE",
    "MAIN OUTCOMES AND MEASURES",
    "DESIGN SETTING AND PARTICIPANTS",
    "DESIGN, SETTING, AND PARTICIPANTS",
    "TRIAL REGISTRATION",
    "CONCLUSIONS AND RELEVANCE",
    "OBJECTIVES",
    "OBJECTIVE",
    "BACKGROUND",
    "IMPORTANCE",
    "METHODS",
    "RESULTS",
    "CONCLUSIONS",
    "CONCLUSION",
    "PURPOSE",
    "AIMS",
    "AIM",
]


def find_heading(sentence):
    text = sentence.lstrip()

    for heading in headings:
        if text.startswith(heading):
            remainder = text[len(heading):]

            # Require a boundary: avoid matching "AIM" in "AIMING".
            if (
                not remainder
                or remainder[0].isspace()
                or remainder[0] in ":.-"
            ):
                return heading

    return None

In [76]:
heading_audit = train_df.copy()
heading_audit["heading_prefix"] = (
    heading_audit["sentence"].apply(find_heading)
)

flagged = heading_audit.loc[
    heading_audit["heading_prefix"].notna()
]

print("Training sentences:", len(heading_audit))
print("Flagged sentences:", len(flagged))
print(f"Flagged percentage: {len(flagged) / len(heading_audit):.2%}")

print("\nCounts by heading:")
print(flagged["heading_prefix"].value_counts())

Training sentences: 180040
Flagged sentences: 42
Flagged percentage: 0.02%

Counts by heading:
heading_prefix
RESULTS                       26
TRIAL REGISTRATION             7
MAIN OUTCOMES AND MEASURES     2
OBJECTIVE                      2
METHODS                        2
CONCLUSIONS AND RELEVANCE      1
CONCLUSION                     1
CONCLUSIONS                    1
Name: count, dtype: int64


In [77]:
display(pd.crosstab(
    flagged["heading_prefix"],
    flagged["label"],
))

label,BACKGROUND,CONCLUSIONS,METHODS,OBJECTIVE,RESULTS
heading_prefix,,,,,
CONCLUSION,0,0,0,0,1
CONCLUSIONS,0,1,0,0,0
CONCLUSIONS AND RELEVANCE,0,0,0,0,1
MAIN OUTCOMES AND MEASURES,0,0,2,0,0
METHODS,0,0,0,2,0
OBJECTIVE,0,0,1,1,0
RESULTS,1,7,12,0,6
TRIAL REGISTRATION,0,6,0,0,1


In [78]:
for _, row in flagged.sample(
    n=min(10, len(flagged)),
    random_state=42,
).iterrows():
    print(f"\nPrefix: {row['heading_prefix']} | Label: {row['label']}")
    print(row["sentence"])



##This detector found too few candidates to make heading removal a priority, although it does not capture every possible heading format.


Prefix: RESULTS | Label: RESULTS
RESULTS for all IIEF and SEP measures were significantly better for TAD-OaD ( p < 0.001 for all ) compared to PBO and were comparable to those observed during PRN-PDE5I treatment .

Prefix: TRIAL REGISTRATION | Label: CONCLUSIONS
TRIAL REGISTRATION clinicaltrials.gov Identifier : NCT00787579 .

Prefix: TRIAL REGISTRATION | Label: CONCLUSIONS
TRIAL REGISTRATION DETAILS : Clinical Trial Registry-India URL : http://ctri.nic.in , Registration Number : CTRI/2009/091 / 001036 .

Prefix: RESULTS | Label: METHODS
RESULTS ; Curative rate of group A was better than that of group B , otherwise incidence rate of critical illness was less than that of group B.

Prefix: RESULTS | Label: BACKGROUND
RESULTS of the feasibility study on the first 100 enrolled patients are presented .

Prefix: OBJECTIVE | Label: METHODS
OBJECTIVE clinical assessments were made by blinded photographic evaluation .

Prefix: METHODS | Label: OBJECTIVE
METHODS : One hundred cases of VCF were

In [79]:
def add_position_feature(df):
    result = df.copy()

    result["total_sentences"] = (
        result.groupby("abstract_id")["sentence"].transform("size")
    )

    # Avoid division by zero for a one-sentence abstract.
    denominator = (result["total_sentences"] - 1).clip(lower=1)

    result["relative_position"] = (
        result["sentence_number"] / denominator
    )

    return result


train_position_df = add_position_feature(train_df)
validation_position_df = add_position_feature(validation_df)

In [80]:
first_id = train_position_df["abstract_id"].iloc[0]

display(
    train_position_df.loc[
        train_position_df["abstract_id"] == first_id,
        ["sentence_number", "total_sentences", "relative_position", "label"],
    ]
)

,sentence_number,total_sentences,relative_position,label
0,0,12,0.000000,OBJECTIVE
1,1,12,0.090909,METHODS
2,2,12,0.181818,METHODS
3,3,12,0.272727,METHODS
4,4,12,0.363636,METHODS
5,5,12,0.454545,METHODS
6,6,12,0.545455,RESULTS
7,7,12,0.636364,RESULTS
8,8,12,0.727273,RESULTS
9,9,12,0.818182,RESULTS


In [81]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd


class PositionDataset(Dataset):
    def __init__(self, dataframe, vocab, label_to_id):
        self.sentences = dataframe["sentence"].tolist()
        self.positions = dataframe["relative_position"].tolist()
        self.labels = dataframe["label"].tolist()
        self.vocab = vocab
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):
        token_ids = [
            self.vocab.get(token, self.vocab["<UNK>"])
            for token in tokenize(self.sentences[index])
        ]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.positions[index], dtype=torch.float32),
            torch.tensor(
                self.label_to_id[self.labels[index]],
                dtype=torch.long,
            ),
        )


def collate_position_batch(batch):
    token_sequences, positions, labels = zip(*batch)

    padded_tokens = pad_sequence(
        token_sequences,
        batch_first=True,
        padding_value=vocab["<PAD>"],
    )

    return (
        padded_tokens,
        torch.stack(positions),
        torch.stack(labels),
    )


position_train_dataset = PositionDataset(
    train_position_df, vocab, label_to_id
)
position_validation_dataset = PositionDataset(
    validation_position_df, vocab, label_to_id
)

position_train_loader = DataLoader(
    position_train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_position_batch,
    generator=torch.Generator().manual_seed(42),
)

position_validation_loader = DataLoader(
    position_validation_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_position_batch,
)

In [82]:
class PositionCNN(nn.Module):
    def __init__(
        self, vocab_size, embedding_dim,
        num_filters, num_classes, pad_id,
    ):
        super().__init__()

        self.pad_id = pad_id

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=pad_id
        )
        self.conv = nn.Conv1d(
            embedding_dim, num_filters,
            kernel_size=3, padding=1,
        )
        self.activation = nn.ReLU()

        # One additional input for relative position.
        self.classifier = nn.Linear(num_filters + 1, num_classes)

    def forward(self, token_ids, positions):
        embeddings = self.embedding(token_ids).transpose(1, 2)
        features = self.activation(self.conv(embeddings))

        mask = (token_ids != self.pad_id).unsqueeze(1)
        features = features.masked_fill(~mask, float("-inf"))

        text_features = features.max(dim=2).values  # [B, 32]
        positions = positions.unsqueeze(1)         # [B, 1]

        combined = torch.cat(
            [text_features, positions], dim=1
        )                                           # [B, 33]

        return self.classifier(combined)             # [B, 5]


torch.manual_seed(42)

position_model = PositionCNN(
    vocab_size=len(vocab),
    embedding_dim=64,
    num_filters=32,
    num_classes=len(label_to_id),
    pad_id=vocab["<PAD>"],
)

position_optimizer = torch.optim.Adam(
    position_model.parameters(), lr=0.001
)
position_loss_fn = nn.CrossEntropyLoss()

In [83]:
def train_position_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = total_correct = total_examples = 0

    for tokens, positions, labels in loader:
        optimizer.zero_grad()

        scores = model(tokens, positions)
        loss = loss_fn(scores, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (scores.argmax(1) == labels).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


def evaluate_position(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    predictions, actual_labels = [], []

    with torch.no_grad():
        for tokens, positions, labels in loader:
            scores = model(tokens, positions)
            loss = loss_fn(scores, labels)

            total_loss += loss.item() * labels.size(0)
            predictions.extend(scores.argmax(1).tolist())
            actual_labels.extend(labels.tolist())

    return (
        total_loss / len(actual_labels),
        accuracy_score(actual_labels, predictions),
        actual_labels,
        predictions,
    )

In [84]:
# checkpoint_dir.mkdir(parents=True, exist_ok=True)

# position_history = []
# best_position_loss = float("inf")

# for epoch in range(1, 6):
#     train_loss, train_accuracy = train_position_epoch(
#         position_model,
#         position_train_loader,
#         position_loss_fn,
#         position_optimizer,
#     )

#     val_loss, val_accuracy, _, _ = evaluate_position(
#         position_model,
#         position_validation_loader,
#         position_loss_fn,
#     )

#     position_history.append({
#         "epoch": epoch,
#         "train_loss": train_loss,
#         "train_accuracy": train_accuracy,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#     })

#     improved = val_loss < best_position_loss
#     if improved:
#         best_position_loss = val_loss

#     position_checkpoint = {
#         "model_state_dict": position_model.state_dict(),
#         "optimizer_state_dict": position_optimizer.state_dict(),
#         "vocab": vocab,
#         "label_to_id": label_to_id,
#         "embedding_dim": 64,
#         "num_filters": 32,
#         "kernel_size": 3,
#         "padding": 1,
#         "pad_id": vocab["<PAD>"],
#         "epochs_completed": epoch,
#         "validation_loss": val_loss,
#         "validation_accuracy": val_accuracy,
#         "best_validation_loss": best_position_loss,
#         "history": position_history,
#         "tokenizer": "lowercase_then_whitespace_split",
#         "position_feature": "sentence_number / max(total_sentences - 1, 1)",
#     }

#     torch.save(
#         position_checkpoint, checkpoint_dir / "cnn_position_latest.pt"
#     )

#     if improved:
#         torch.save(
#             position_checkpoint, checkpoint_dir / "cnn_position_best.pt"
#         )

#     print(
#         f"Epoch {epoch} | "
#         f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.2%} | "
#         f"Validation loss: {val_loss:.4f}, accuracy: {val_accuracy:.2%}"
#         + (" | Best saved" if improved else ""),
#         flush=True,
#     )

In [85]:
best_position_checkpoint = torch.load(
    checkpoint_dir / "cnn_position_best.pt",
    map_location="cpu",
    weights_only=True,
)

position_model.load_state_dict(
    best_position_checkpoint["model_state_dict"]
)

_, position_accuracy, position_labels, position_predictions = (
    evaluate_position(
        position_model,
        position_validation_loader,
        position_loss_fn,
    )
)

class_ids = list(range(len(label_to_id)))
id_to_label = {i: label for label, i in label_to_id.items()}
class_names = [id_to_label[i] for i in class_ids]

print("Selected epoch:", best_position_checkpoint["epochs_completed"])

print(classification_report(
    position_labels,
    position_predictions,
    labels=class_ids,
    target_names=class_names,
    digits=3,
    zero_division=0,
))

position_macro_f1 = f1_score(
    position_labels,
    position_predictions,
    labels=class_ids,
    average="macro",
    zero_division=0,
)

comparison = pd.DataFrame({
    "Model": ["Original CNN", "CNN + relative position"],
    "Validation accuracy (%)": [81.11, position_accuracy * 100],
    "Validation macro-F1 (%)": [74.6, position_macro_f1 * 100],
})

display(comparison)

Selected epoch: 4
              precision    recall  f1-score   support

  BACKGROUND      0.686     0.778     0.729      3449
   OBJECTIVE      0.725     0.524     0.608      2376
     METHODS      0.888     0.935     0.911      9964
     RESULTS      0.908     0.876     0.892      9841
 CONCLUSIONS      0.867     0.871     0.869      4582

    accuracy                          0.856     30212
   macro avg      0.815     0.797     0.802     30212
weighted avg      0.856     0.856     0.854     30212



,Model,Validation accuracy (%),Validation macro-F1 (%)
0,Original CNN,81.110000,74.600000
1,CNN + relative position,85.581888,80.184486


In [86]:
from sklearn.metrics import confusion_matrix

_, _, actual_labels, predictions = evaluate_position(
    position_model,
    position_validation_loader,
    position_loss_fn,
)

label_ids = sorted(id_to_label)
label_names = [id_to_label[i] for i in label_ids]

cm = confusion_matrix(
    actual_labels,
    predictions,
    labels=label_ids,
)

confusion_df = pd.DataFrame(
    cm,
    index=pd.Index(label_names, name="Actual"),
    columns=pd.Index(label_names, name="Predicted"),
)

display(confusion_df)

Predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
Actual,,,,,
BACKGROUND,2684,428,234,26,77
OBJECTIVE,927,1244,191,12,2
METHODS,190,41,9317,369,47
RESULTS,44,1,688,8619,489
CONCLUSIONS,65,1,60,464,3992


#SENTENCE POSITION

Adding relative sentence position improved validation accuracy from 81.11% to 85.58% and macro-F1 from 0.746 to 0.802. Errors between BACKGROUND and CONCLUSIONS decreased substantially, while BACKGROUND–OBJECTIVE confusion persisted.

In [87]:

###Samples three OBJECTIVE → BACKGROUND errors and prints each full abstract, including position and predictions.

# Predictions from your restored best position model
_, _, actual_labels, predictions = evaluate_position(
    position_model,
    position_validation_loader,
    position_loss_fn,
)

results = validation_position_df.copy().reset_index(drop=True)

assert len(results) == len(predictions)
assert results["label"].tolist() == [
    id_to_label[i] for i in actual_labels
]

results["predicted_label"] = [
    id_to_label[i] for i in predictions
]

# Find actual OBJECTIVE sentences predicted as BACKGROUND
errors = results[
    (results["label"] == "OBJECTIVE")
    & (results["predicted_label"] == "BACKGROUND")
]

print(f"Matching errors: {len(errors)}")

examples = errors.sample(
    n=min(3, len(errors)),
    random_state=42,
)

for row_index, example in examples.iterrows():
    print("\n" + "=" * 80)
    print(f"Abstract ID: {example['abstract_id']}")
    print(f"Misclassified sentence: {example['sentence']}")
    print(f"Relative position: {example['relative_position']:.3f}")
    print("\nFull abstract:")

    abstract = results[
        results["abstract_id"] == example["abstract_id"]
    ].sort_values("sentence_number")

    for index, row in abstract.iterrows():
        marker = ">>>" if index == row_index else "   "

        print(
            f"\n{marker} Sentence {row['sentence_number']} "
            f"| Position: {row['relative_position']:.3f} "
            f"| Actual: {row['label']} "
            f"| Predicted: {row['predicted_label']}"
        )
        print(row["sentence"])

Matching errors: 927

Abstract ID: 25200821
Misclassified sentence: Tubal factor accounts for 25-30 % of cases of female infertility .
Relative position: 0.000

Full abstract:

>>> Sentence 0 | Position: 0.000 | Actual: OBJECTIVE | Predicted: BACKGROUND
Tubal factor accounts for 25-30 % of cases of female infertility .

    Sentence 1 | Position: 0.111 | Actual: OBJECTIVE | Predicted: BACKGROUND
Laparoscopy `` Gold Standard '' for tubal evaluation .

    Sentence 2 | Position: 0.222 | Actual: OBJECTIVE | Predicted: BACKGROUND
However , it is known that during the initial infection of the fallopian tube mucosal damage occurs , a condition which plays a decisive role in reproduction .

    Sentence 3 | Position: 0.333 | Actual: METHODS | Predicted: METHODS
In this prospective randomized study , 468 infertile women with evidence of fallopian tube disease were included .

    Sentence 4 | Position: 0.444 | Actual: METHODS | Predicted: METHODS
In this , for 256 patients ( group 1 ) after la

Three sampled OBJECTIVE → BACKGROUND errors involved introductory statements that read as background information. In two abstracts, the model correctly classified the subsequent explicit aim sentence. This suggests that section-level labelling may contribute to the remaining confusion.

| Abstract | Highlighted sentence’s role           | What else we observe                                                  |
| -------- | ------------------------------------- | --------------------------------------------------------------------- |
| 25200821 | Existing knowledge about infertility  | All three introductory sentences are labelled OBJECTIVE               |
| 24449733 | Established use of a diagnostic tool  | The model correctly identifies “The aim of this study… (Sentence 2)” as OBJECTIVE  |
| 25116277 | Current treatment and an evidence gap | The model correctly identifies the next sentence’s explicit study aim |


In [88]:
import torch
from torch import nn
from sklearn.metrics import classification_report

best_path = checkpoint_dir / "cnn_position_reused_best.pt"
latest_path = checkpoint_dir / "cnn_position_reused_latest.pt"

# Resume fine-tuning when a latest checkpoint exists.
resuming = latest_path.exists()

source_path = (
    latest_path if resuming
    else checkpoint_dir / "cnn_best.pt"
)

checkpoint = torch.load(
    source_path,
    map_location="cpu",
    weights_only=True,
)

# Ensure the current data uses the checkpoint's token and label mappings.
assert vocab == checkpoint["vocab"], "Vocabulary does not match checkpoint."
assert label_to_id == checkpoint["label_to_id"], "Labels do not match checkpoint."

# Retain this name because the checkpoint-saving function uses it.
original_checkpoint = checkpoint

reused_model = PositionCNN(
    vocab_size=len(vocab),
    embedding_dim=checkpoint["embedding_dim"],
    num_filters=checkpoint["num_filters"],
    num_classes=len(label_to_id),
    pad_id=checkpoint["pad_id"],
)

if resuming:
    # Restore all parameters, including the learned position weights.
    reused_model.load_state_dict(checkpoint["model_state_dict"])

else:
    # Initialize from the original text-only CNN.
    old_state = checkpoint["model_state_dict"]

    reused_model.embedding.load_state_dict({
        "weight": old_state["embedding.weight"],
    })

    reused_model.conv.load_state_dict({
        "weight": old_state["conv.weight"],
        "bias": old_state["conv.bias"],
    })

    # Copy text weights and initially give position zero influence.
    with torch.no_grad():
        reused_model.classifier.weight[:, :-1].copy_(
            old_state["classifier.weight"]
        )
        reused_model.classifier.weight[:, -1].zero_()
        reused_model.classifier.bias.copy_(
            old_state["classifier.bias"]
        )

reused_optimizer = torch.optim.Adam(
    reused_model.parameters(),
    lr=0.001,
)
reused_loss_fn = nn.CrossEntropyLoss()

if resuming:
    # Restore Adam's accumulated state and training progress.
    reused_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    history = checkpoint["history"]
    best_loss = checkpoint["best_validation_loss"]
    start_epoch = checkpoint["fine_tuning_epochs_completed"] + 1

    print(f"Resuming fine-tuning from epoch {start_epoch}.")

else:
    # Evaluate the starting model before learning to use position.
    initial_loss, initial_accuracy, _, _ = evaluate_position(
        reused_model,
        position_validation_loader,
        reused_loss_fn,
    )

    history = []
    best_loss = initial_loss
    start_epoch = 1

    print("Initialized from the original CNN.")
    print(f"Initial validation loss: {initial_loss:.4f}")
    print(f"Initial validation accuracy: {initial_accuracy:.2%}")

Resuming fine-tuning from epoch 2.


In [89]:
# Evaluate before any updates: position currently contributes zero.
initial_loss, initial_accuracy, _, _ = evaluate_position(
    reused_model,
    position_validation_loader,
    reused_loss_fn,
)

print(f"Before fine-tuning | Validation loss: {initial_loss:.4f}")
print(f"Before fine-tuning | Validation accuracy: {initial_accuracy:.2%}")

Before fine-tuning | Validation loss: 0.4452
Before fine-tuning | Validation accuracy: 83.74%


In [90]:
# Save the starting state so it remains available if training worsens validation.
#best_loss = initial_loss
#history = []

best_path = checkpoint_dir / "cnn_position_reused_best.pt"
latest_path = checkpoint_dir / "cnn_position_reused_latest.pt"


def make_reused_checkpoint(epoch, val_loss, val_accuracy):
    # Store weights and metadata needed to reconstruct or resume this experiment.
    return {
        "model_state_dict": reused_model.state_dict(),
        "optimizer_state_dict": reused_optimizer.state_dict(),
        "vocab": vocab,
        "label_to_id": label_to_id,
        "embedding_dim": original_checkpoint["embedding_dim"],
        "num_filters": original_checkpoint["num_filters"],
        "kernel_size": 3,
        "padding": 1,
        "pad_id": vocab["<PAD>"],
        "fine_tuning_epochs_completed": epoch,
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "best_validation_loss": best_loss,
        "history": history,
        "tokenizer": "lowercase_then_whitespace_split",
        "position_feature": "sentence_number / max(total_sentences - 1, 1)",
        "initialized_from": "cnn_best.pt",
    }


# torch.save(
#     make_reused_checkpoint(0, initial_loss, initial_accuracy),
#     best_path,
# )

# for epoch in range(1, 6):
# Save epoch 0 only when starting a new fine-tuning experiment.
if not resuming:
    starting_checkpoint = make_reused_checkpoint(
        0, initial_loss, initial_accuracy
    )
    torch.save(starting_checkpoint, best_path)
    torch.save(starting_checkpoint, latest_path)

# Five total fine-tuning epochs, including any already completed.
for epoch in range(start_epoch, 6):
    # Keep your existing training loop body here.
    train_loss, train_accuracy = train_position_epoch(
        reused_model,
        position_train_loader,
        reused_loss_fn,
        reused_optimizer,
    )

    val_loss, val_accuracy, _, _ = evaluate_position(
        reused_model,
        position_validation_loader,
        reused_loss_fn,
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
    })

    improved = val_loss < best_loss
    if improved:
        best_loss = val_loss

    checkpoint = make_reused_checkpoint(epoch, val_loss, val_accuracy)
    torch.save(checkpoint, latest_path)

    if improved:
        torch.save(checkpoint, best_path)

    print(
        f"Fine-tuning epoch {epoch} | "
        f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.2%} | "
        f"Validation loss: {val_loss:.4f}, accuracy: {val_accuracy:.2%}"
        + (" | Best saved" if improved else "")
    )

Fine-tuning epoch 2 | Train loss: 0.3023, accuracy: 89.33% | Validation loss: 0.4302, accuracy: 84.28% | Best saved
Fine-tuning epoch 3 | Train loss: 0.2530, accuracy: 91.15% | Validation loss: 0.4420, accuracy: 83.88%
Fine-tuning epoch 4 | Train loss: 0.2108, accuracy: 92.75% | Validation loss: 0.4677, accuracy: 83.65%
Fine-tuning epoch 5 | Train loss: 0.1735, accuracy: 94.22% | Validation loss: 0.4946, accuracy: 83.08%


In [91]:
# Restore the checkpoint with the lowest validation loss.
best_checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=True,
)
reused_model.load_state_dict(best_checkpoint["model_state_dict"])

_, _, actual_labels, predictions = evaluate_position(
    reused_model,
    position_validation_loader,
    reused_loss_fn,
)

print(
    "Selected fine-tuning epoch:",
    best_checkpoint["fine_tuning_epochs_completed"],
)

label_ids = sorted(id_to_label)

print(classification_report(
    actual_labels,
    predictions,
    labels=label_ids,
    target_names=[id_to_label[i] for i in label_ids],
    digits=3,
    zero_division=0,
))

Selected fine-tuning epoch: 2
              precision    recall  f1-score   support

  BACKGROUND      0.684     0.702     0.693      3449
   OBJECTIVE      0.678     0.575     0.622      2376
     METHODS      0.885     0.926     0.905      9964
     RESULTS      0.895     0.873     0.884      9841
 CONCLUSIONS      0.833     0.842     0.837      4582

    accuracy                          0.843     30212
   macro avg      0.795     0.784     0.788     30212
weighted avg      0.841     0.843     0.841     30212

